# Build a Diffusion Model: 2D Spiral

In this notebook we build a **Denoising Diffusion Probabilistic Model (DDPM)** from scratch and train it to generate 2D spiral data.

Everything runs on CPU in under a minute — no GPU needed.

This notebook mirrors the diffusion lesson on the [mlearn website](https://mlearn.org). Head there for the full explanation of the math and intuition behind each step.

---

**How to use this notebook:** Each section has an **exercise cell** where you write code yourself, followed by a **collapsed solution cell** you can expand if you get stuck. Try the exercise first!

## 1 — Setup

In [ ]:
!pip install torch matplotlib -q

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch {torch.__version__}")

## 2 — The Spiral Dataset

We generate points that lie on a 2D spiral. This is the distribution our diffusion model will learn to sample from.

### Exercise 1: Generate the Spiral

Write a function `make_spiral` that generates `n_points` 2D points lying on a spiral.

**Recipe:**
1. Create a parameter `t` that goes from `0` to `4π` (use `torch.linspace`).
2. Compute a radius `r = t / (4π)` so the spiral expands outward.
3. Convert to Cartesian: `x = r * cos(t)`, `y = r * sin(t)`, plus a little Gaussian noise.
4. Stack into a `(n_points, 2)` tensor.

Then create the dataset and plot it with `plt.scatter`.

In [ ]:
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate 2D spiral dataset."""
    # Step 1: Create parameter t from 0 to 4*pi
    # YOUR CODE HERE

    # Step 2: Radius grows linearly with t
    # YOUR CODE HERE

    # Step 3: Convert to (x, y) with some noise
    # Hint: x = r * cos(t) + noise * randn * 0.1
    # YOUR CODE HERE

    # Step 4: Stack into (n_points, 2) tensor
    # YOUR CODE HERE
    pass

dataset = make_spiral()

# Plot the spiral
plt.figure(figsize=(5, 5))
# YOUR CODE HERE: scatter plot of dataset
plt.title("Spiral Dataset")
plt.axis("equal")
plt.show()

In [ ]:
#@title Solution (click to expand)
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate 2D spiral dataset."""
    t = torch.linspace(0, 4 * np.pi, n_points)
    r = t / (4 * np.pi)  # radius grows linearly
    x = r * torch.cos(t) + noise * torch.randn(n_points) * 0.1
    y = r * torch.sin(t) + noise * torch.randn(n_points) * 0.1
    data = torch.stack([x, y], dim=1)  # (n_points, 2)
    return data

dataset = make_spiral()

plt.figure(figsize=(5, 5))
plt.scatter(dataset[:, 0], dataset[:, 1], s=3, alpha=0.5)
plt.title("Spiral Dataset")
plt.axis("equal")
plt.show()

## 3 — Forward Process (Adding Noise)

The forward process gradually destroys structure by adding Gaussian noise according to a variance schedule.

Given clean data $x_0$, we can jump to any timestep $t$ in closed form:

$$q(x_t | x_0) = \mathcal{N}(x_t;\; \sqrt{\bar\alpha_t}\, x_0,\; (1 - \bar\alpha_t)\, I)$$

### Exercise 2: Noise Schedule and Forward Diffusion

Build the noise schedule and the forward diffusion function.

**Step A — Noise schedule:**
- `betas`: linearly spaced from `1e-4` to `0.02`, length `T=300`
- `alphas = 1 - betas`
- `alpha_bars`: cumulative product of `alphas` (use `torch.cumprod`)

**Step B — `forward_diffusion(x_0, t, noise)`:**
- If `noise` is None, sample from standard normal
- Look up `alpha_bars[t]` and reshape to `(batch, 1)` for broadcasting
- Compute: `x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise`
- Return `(x_t, noise)`

In [ ]:
# Noise schedule
T = 300  # total diffusion timesteps

# Step A: Build the schedule
# betas = ...           # linear schedule from 1e-4 to 0.02
# alphas = ...          # 1 - betas
# alpha_bars = ...      # cumulative product of alphas
# YOUR CODE HERE


def forward_diffusion(
    x_0: torch.Tensor,
    t: torch.Tensor,
    noise: Optional[torch.Tensor] = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Apply forward diffusion to x_0 at timesteps t.
    Returns (x_t, noise).
    """
    # Step B: Implement the closed-form forward diffusion
    # Hint: ab = alpha_bars[t].unsqueeze(-1) for shape (batch, 1)
    # YOUR CODE HERE
    pass

In [ ]:
#@title Solution (click to expand)
# Noise schedule
T = 300  # total diffusion timesteps

betas = torch.linspace(1e-4, 0.02, T)          # linear schedule
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)       # cumulative product

def forward_diffusion(
    x_0: torch.Tensor,
    t: torch.Tensor,
    noise: Optional[torch.Tensor] = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Apply forward diffusion to x_0 at timesteps t.
    Returns (x_t, noise).
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    ab = alpha_bars[t].unsqueeze(-1)  # (batch, 1)
    x_t = torch.sqrt(ab) * x_0 + torch.sqrt(1.0 - ab) * noise
    return x_t, noise

### Visualize the Forward Process

Run this cell to see the spiral progressively destroyed by noise.

In [ ]:
# Visualize the forward process at several timesteps
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
steps_to_show = [0, 50, 100, 200, 299]

for ax, step in zip(axes, steps_to_show):
    t_vec = torch.full((dataset.shape[0],), step, dtype=torch.long)
    x_t, _ = forward_diffusion(dataset, t_vec)
    ax.scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), s=2, alpha=0.4)
    ax.set_title(f"t = {step}")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")

plt.suptitle("Forward Diffusion: Spiral \u2192 Noise", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 4 — The Noise Predictor

We train a small MLP that takes a noisy point $x_t$ and the timestep $t$, and predicts the noise $\epsilon$ that was added.

The timestep is encoded with **sinusoidal positional embeddings** (same idea as in Transformers).

### Exercise 3: Sinusoidal Embedding and Noise Predictor Network

This is one of the trickiest parts. You need to build two things:

**A) `sinusoidal_embedding(t, dim)`** — Maps integer timesteps to a vector.
- Compute `half = dim // 2` frequency bands
- `freqs = exp(-log(10000) * arange(half) / half)`
- Multiply: `args = t_float[:, None] * freqs[None, :]` giving shape `(batch, half)`
- Concatenate `sin(args)` and `cos(args)` along the last dimension → `(batch, dim)`

**B) `NoisePredictor(nn.Module)`** — An MLP that predicts 2D noise.
- `__init__`: Create an `nn.Sequential` with:
  - `Linear(2 + time_dim, hidden_dim)` → `ReLU` → `Linear(hidden_dim, hidden_dim)` → `ReLU` → `Linear(hidden_dim, 2)`
- `forward(x_t, t)`:
  1. Compute time embedding: `t_emb = sinusoidal_embedding(t, self.time_dim)` → `(batch, time_dim)`
  2. Concatenate `[x_t, t_emb]` along dim=-1 → `(batch, 2 + time_dim)`
  3. Pass through `self.net`

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int = 64) -> torch.Tensor:
    """
    Map integer timesteps to sinusoidal embeddings of size `dim`.
    t: (batch,) long tensor
    returns: (batch, dim)
    """
    half = dim // 2

    # Compute frequency bands
    # freqs = exp(-log(10000) * arange(half) / half)
    # YOUR CODE HERE

    # Compute args = t_float[:, None] * freqs[None, :]
    # YOUR CODE HERE

    # Return concatenation of sin(args) and cos(args)
    # YOUR CODE HERE
    pass


class NoisePredictor(nn.Module):
    def __init__(self, hidden_dim: int = 128, time_dim: int = 64):
        super().__init__()
        self.time_dim = time_dim

        # Build the MLP: input is [x_t (2 dims), time_emb (time_dim dims)]
        # Architecture: Linear -> ReLU -> Linear -> ReLU -> Linear
        # Input size:  2 + time_dim
        # Hidden size: hidden_dim
        # Output size: 2 (predicting 2D noise)
        self.net = nn.Sequential(
            # YOUR CODE HERE
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # Step 1: Get time embedding
        # t_emb = sinusoidal_embedding(t, self.time_dim)
        # YOUR CODE HERE

        # Step 2: Concatenate x_t and t_emb along dim=-1
        # YOUR CODE HERE

        # Step 3: Pass through self.net and return
        # YOUR CODE HERE
        pass


model = NoisePredictor()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title Solution (click to expand)
def sinusoidal_embedding(t: torch.Tensor, dim: int = 64) -> torch.Tensor:
    """
    Map integer timesteps to sinusoidal embeddings of size `dim`.
    t: (batch,) long tensor
    returns: (batch, dim)
    """
    half = dim // 2
    freqs = torch.exp(
        -np.log(10000.0) * torch.arange(half, dtype=torch.float32) / half
    )
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)  # (batch, half)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (batch, dim)


class NoisePredictor(nn.Module):
    def __init__(self, hidden_dim: int = 128, time_dim: int = 64):
        super().__init__()
        self.time_dim = time_dim
        # Project concatenated [x_t (2), time_emb (time_dim)] into hidden
        self.net = nn.Sequential(
            nn.Linear(2 + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),  # predict 2D noise
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = sinusoidal_embedding(t, self.time_dim)  # (batch, time_dim)
        inp = torch.cat([x_t, t_emb], dim=-1)           # (batch, 2 + time_dim)
        return self.net(inp)                              # (batch, 2)


model = NoisePredictor()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5 — Training

Each training step:
1. Sample a batch of clean points $x_0$ from the dataset.
2. Sample random timesteps $t \sim \text{Uniform}\{0, \ldots, T{-}1\}$.
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$.
4. Compute the noisy version $x_t$ via the forward process.
5. Predict $\hat\epsilon = f_\theta(x_t, t)$.
6. Minimize $\|\epsilon - \hat\epsilon\|^2$.

### Exercise 4: Training Loop

Implement the training loop following the 6 steps above.

**Settings:** `Adam` optimizer with `lr=1e-3`, `batch_size=256`, `n_steps=5000`.

**Inside each step:**
1. Sample random indices into `dataset` and grab `x_0`
2. Sample random timesteps `t` in `[0, T)`
3. Sample noise from `torch.randn_like(x_0)`
4. Compute `x_t` via `forward_diffusion`
5. Get `noise_pred = model(x_t, t)`
6. Compute MSE loss between `noise_pred` and `noise`, backprop, step

Print loss every 1000 steps and plot the loss curve at the end.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 256
n_steps = 5000
losses = []

for step in range(n_steps):
    # 1. Sample batch of clean points
    # YOUR CODE HERE

    # 2. Sample random timesteps
    # YOUR CODE HERE

    # 3. Sample noise
    # YOUR CODE HERE

    # 4. Forward diffusion to get x_t
    # YOUR CODE HERE

    # 5. Predict noise with model
    # YOUR CODE HERE

    # 6. MSE loss, backward, optimizer step
    # YOUR CODE HERE

    losses.append(loss.item())
    if (step + 1) % 1000 == 0:
        print(f"Step {step+1:5d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, linewidth=0.5)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
#@title Solution (click to expand)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 256
n_steps = 5000
losses = []

for step in range(n_steps):
    # 1. Sample batch
    idx = torch.randint(0, len(dataset), (batch_size,))
    x_0 = dataset[idx]  # (batch, 2)

    # 2. Sample random timesteps
    t = torch.randint(0, T, (batch_size,))

    # 3. Sample noise
    noise = torch.randn_like(x_0)

    # 4. Forward diffusion
    x_t, _ = forward_diffusion(x_0, t, noise)

    # 5. Predict noise
    noise_pred = model(x_t, t)

    # 6. MSE loss
    loss = nn.functional.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if (step + 1) % 1000 == 0:
        print(f"Step {step+1:5d} | Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, linewidth=0.5)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()

## 6 — Sampling (Reverse Process)

Starting from pure noise $x_T \sim \mathcal{N}(0, I)$, we iteratively denoise using the DDPM update rule:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\,\hat\epsilon_\theta(x_t, t)\right) + \sigma_t\, z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$.

### Exercise 5: DDPM Sampling Loop

This is the other tricky part. Implement the reverse sampling loop.

**Function signature:** `ddpm_sample(model, n_samples=2000, record_steps=None)`

**Algorithm:**
1. Start from pure noise: `x = torch.randn(n_samples, 2)`
2. Loop `t_idx` from `T-1` down to `0`:
   - Create a batch of timesteps: `t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)`
   - Predict noise: `eps_pred = model(x, t_batch)`
   - Look up `beta_t`, `alpha_t`, `alpha_bar_t` for this timestep
   - Compute the mean: `mean = (1/sqrt(alpha_t)) * (x - (beta_t / sqrt(1 - alpha_bar_t)) * eps_pred)`
   - If `t_idx > 0`: add noise `x = mean + sqrt(beta_t) * randn_like(x)`
   - If `t_idx == 0`: `x = mean` (no noise at the last step)
   - Optionally record snapshots
3. Return `(x, snapshots)`

**Important:** Wrap the function with `@torch.no_grad()` since we are doing inference.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples: int = 2000, record_steps: list = None):
    """
    DDPM reverse sampling.
    Returns final samples and (optionally) snapshots at recorded steps.
    """
    # Step 1: Start from pure noise
    x = torch.randn(n_samples, 2)
    snapshots = {}

    # Step 2: Reverse loop from T-1 down to 0
    for t_idx in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)

        # Predict noise
        # eps_pred = ...
        # YOUR CODE HERE

        # Look up schedule values for this timestep
        # beta_t = betas[t_idx]
        # alpha_t = alphas[t_idx]
        # alpha_bar_t = alpha_bars[t_idx]
        # YOUR CODE HERE

        # Compute the denoised mean
        # mean = (1 / sqrt(alpha_t)) * (x - (beta_t / sqrt(1 - alpha_bar_t)) * eps_pred)
        # YOUR CODE HERE

        # Add noise if t_idx > 0, otherwise x = mean
        # YOUR CODE HERE

        # Record snapshot if requested
        if record_steps is not None and t_idx in record_steps:
            snapshots[t_idx] = x.clone()

    return x, snapshots


# Test: sample and visualize
record_at = [299, 200, 100, 50, 0]
samples, snaps = ddpm_sample(model, n_samples=2000, record_steps=record_at)

fig, axes = plt.subplots(1, len(record_at), figsize=(20, 4))
for ax, t_idx in zip(axes, record_at):
    pts = snaps[t_idx].numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t_idx}")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")

plt.suptitle("Reverse Process: Noise \u2192 Spiral", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#@title Solution (click to expand)
@torch.no_grad()
def ddpm_sample(model, n_samples: int = 2000, record_steps: list = None):
    """
    DDPM reverse sampling.
    Returns final samples and (optionally) snapshots at recorded steps.
    """
    x = torch.randn(n_samples, 2)  # start from pure noise
    snapshots = {}

    for t_idx in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)

        # Predict noise
        eps_pred = model(x, t_batch)

        # DDPM update
        beta_t = betas[t_idx]
        alpha_t = alphas[t_idx]
        alpha_bar_t = alpha_bars[t_idx]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )

        if t_idx > 0:
            sigma = torch.sqrt(beta_t)
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean

        if record_steps is not None and t_idx in record_steps:
            snapshots[t_idx] = x.clone()

    return x, snapshots


# Sample and visualize intermediate steps
record_at = [299, 200, 100, 50, 0]
samples, snaps = ddpm_sample(model, n_samples=2000, record_steps=record_at)

fig, axes = plt.subplots(1, len(record_at), figsize=(20, 4))
for ax, t_idx in zip(axes, record_at):
    pts = snaps[t_idx].numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t_idx}")
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")

plt.suptitle("Reverse Process: Noise \u2192 Spiral", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Compare Real vs Generated

Run this cell to see the real data side-by-side with the generated samples.

In [ ]:
# Compare generated samples to real data
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].scatter(dataset[:, 0].numpy(), dataset[:, 1].numpy(), s=2, alpha=0.4)
axes[0].set_title("Real Data")
axes[0].set_xlim(-2, 2)
axes[0].set_ylim(-2, 2)
axes[0].set_aspect("equal")

axes[1].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=2, alpha=0.4, color="tab:orange")
axes[1].set_title("Generated Samples")
axes[1].set_xlim(-2, 2)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect("equal")

plt.suptitle("Real vs Generated", fontsize=14)
plt.tight_layout()
plt.show()

## 7 — Experiments: Other 2D Datasets

The same model architecture and training loop work on other shapes. Let's try **moons**, **circles**, and a **figure-8**.

### Exercise 6: Alternative Dataset Generators

Write three dataset generators. Each returns a `(n_points, 2)` tensor.

**A) `make_moons`** — Two interleaving half-circles:
- `theta` from `0` to `pi`, `n = n_points // 2`
- Top moon: `(cos(theta), sin(theta))`
- Bottom moon: `(1 - cos(theta), 1 - sin(theta) - 0.5)`
- Concatenate, add noise, center by subtracting the mean

**B) `make_circles`** — Two concentric circles:
- `theta` from `0` to `2*pi`, `n = n_points // 2`
- Outer: `(cos(theta), sin(theta))`
- Inner: `0.5 * (cos(theta), sin(theta))`
- Concatenate, add noise

**C) `make_figure8`** — Lemniscate of Bernoulli:
- `t` from `0` to `2*pi`
- `x = cos(t) / (1 + sin(t)^2)`
- `y = sin(t) * cos(t) / (1 + sin(t)^2)`
- Add noise

In [ ]:
def make_moons(n_points: int = 2000, noise: float = 0.05) -> torch.Tensor:
    """Two interleaving half-circles."""
    n = n_points // 2
    # YOUR CODE HERE
    pass


def make_circles(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Two concentric circles."""
    n = n_points // 2
    # YOUR CODE HERE
    pass


def make_figure8(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Lemniscate of Bernoulli (figure-8)."""
    # YOUR CODE HERE
    pass


# Preview all datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, gen) in zip(axes, [("Moons", make_moons), ("Circles", make_circles), ("Figure-8", make_figure8)]):
    d = gen()
    ax.scatter(d[:, 0].numpy(), d[:, 1].numpy(), s=2, alpha=0.5)
    ax.set_title(name)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
#@title Solution (click to expand)
def make_moons(n_points: int = 2000, noise: float = 0.05) -> torch.Tensor:
    """Two interleaving half-circles."""
    n = n_points // 2
    theta = torch.linspace(0, np.pi, n)
    x_top = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    x_bot = torch.stack([1.0 - torch.cos(theta), 1.0 - torch.sin(theta) - 0.5], dim=1)
    data = torch.cat([x_top, x_bot], dim=0)
    data += noise * torch.randn_like(data)
    # center
    data -= data.mean(dim=0)
    return data


def make_circles(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Two concentric circles."""
    n = n_points // 2
    theta = torch.linspace(0, 2 * np.pi, n)
    outer = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    inner = 0.5 * torch.stack([torch.cos(theta), torch.sin(theta)], dim=1)
    data = torch.cat([outer, inner], dim=0)
    data += noise * torch.randn_like(data)
    return data


def make_figure8(n_points: int = 2000, noise: float = 0.03) -> torch.Tensor:
    """Lemniscate of Bernoulli (figure-8)."""
    t = torch.linspace(0, 2 * np.pi, n_points)
    scale = 1.0
    x = scale * torch.cos(t) / (1 + torch.sin(t) ** 2)
    y = scale * torch.sin(t) * torch.cos(t) / (1 + torch.sin(t) ** 2)
    data = torch.stack([x, y], dim=1)
    data += noise * torch.randn_like(data)
    return data


# Preview all datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, gen) in zip(axes, [("Moons", make_moons), ("Circles", make_circles), ("Figure-8", make_figure8)]):
    d = gen()
    ax.scatter(d[:, 0].numpy(), d[:, 1].numpy(), s=2, alpha=0.5)
    ax.set_title(name)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Train and Sample on All Datasets

Run this cell to train a fresh model on each dataset and compare real vs generated.

In [ ]:
def train_and_sample(data: torch.Tensor, name: str, n_steps: int = 5000):
    """Train a fresh model on `data` and show generated samples."""
    m = NoisePredictor()
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    bs = 256

    for step in range(n_steps):
        idx = torch.randint(0, len(data), (bs,))
        x_0 = data[idx]
        t = torch.randint(0, T, (bs,))
        noise = torch.randn_like(x_0)
        x_t, _ = forward_diffusion(x_0, t, noise)
        loss = nn.functional.mse_loss(m(x_t, t), noise)
        opt.zero_grad()
        loss.backward()
        opt.step()

    # Sample
    samples, _ = ddpm_sample(m, n_samples=2000)
    return samples


datasets_to_try = [
    ("Moons", make_moons()),
    ("Circles", make_circles()),
    ("Figure-8", make_figure8()),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col, (name, data) in enumerate(datasets_to_try):
    print(f"Training on {name}...")
    generated = train_and_sample(data, name)

    # Real
    axes[0, col].scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=2, alpha=0.4)
    axes[0, col].set_title(f"{name} (Real)")
    axes[0, col].set_aspect("equal")
    axes[0, col].set_xlim(-2, 2)
    axes[0, col].set_ylim(-2, 2)

    # Generated
    axes[1, col].scatter(generated[:, 0].numpy(), generated[:, 1].numpy(), s=2, alpha=0.4, color="tab:orange")
    axes[1, col].set_title(f"{name} (Generated)")
    axes[1, col].set_aspect("equal")
    axes[1, col].set_xlim(-2, 2)
    axes[1, col].set_ylim(-2, 2)

plt.suptitle("Experiments: Real vs Generated", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

That's it! With just an MLP and a few lines of code, DDPM learns to reverse the noise process and generate samples from arbitrary 2D distributions.

For the full explanation of the math, variance schedules, and connections to score matching, check out the [mlearn diffusion lesson](https://mlearn.org).